# XGBoost with accessibility features

This notebook cleans the rental data, joins geocoded HDB blocks, calculates nearest school/mall/MRT distances in metres, encodes the selected features, and evaluates XGBoost.

## Load and clean

Exact duplicate rental rows are removed before joining. Addresses are normalized identically in both tables so the HDB join is deterministic. The HDB reference supplies block latitude and longitude.

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HERE=Path.cwd().resolve(); ROOT=next(p for p in [HERE,*HERE.parents] if (p/'data'/'train.csv').exists())
rent_raw=pd.read_csv(ROOT/'data/train.csv'); hdb_raw=pd.read_csv(ROOT/'data/sg-hdb-block.csv',dtype={'POSTAL_CODE':'string'})
schools_raw=pd.read_csv(ROOT/'data/sg-schools.csv'); malls_raw=pd.read_csv(ROOT/'data/sg-shopping-malls.csv'); mrt_raw=pd.read_csv(ROOT/'data/sg-mrt-stations.csv')
replacements={r'\bavenue\b':'ave',r'\broad\b':'rd',r'\bstreet\b':'st',r'\bdrive\b':'dr',r'\bcentral\b':'ctrl',r'\bplace\b':'pl',r'\bclose\b':'cl',r'\bcrescent\b':'cres',r'\blane\b':'ln',r'\bview\b':'vw'}
def address_key(df):
    out=df.copy()
    for c in ['TOWN','BLOCK','STREET']: out[c]=out[c].astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
    for pattern,replacement in replacements.items(): out['STREET']=out['STREET'].str.replace(pattern,replacement,regex=True)
    out['ADDRESS_KEY']=out['TOWN']+'|'+out['BLOCK']+'|'+out['STREET']; return out
rent=address_key(rent_raw); print('exact duplicate rows:',int(rent.duplicated(keep=False).sum())); rent=rent.drop_duplicates().reset_index(drop=True)
hdb=address_key(hdb_raw).drop_duplicates('ADDRESS_KEY')
rent['RENT_APPROVAL_DATE']=pd.to_datetime(rent['RENT_APPROVAL_DATE'],format='%Y-%m',errors='coerce'); rent['RENT_YEAR']=rent['RENT_APPROVAL_DATE'].dt.year; rent['RENT_MONTH']=rent['RENT_APPROVAL_DATE'].dt.month
rent_geo=rent.merge(hdb[['ADDRESS_KEY','POSTAL_CODE','LATITUDE','LONGITUDE','MAX_FLOOR','YEAR_COMPLETED','SUBZONE','PLANNING_AREA','REGION']],on='ADDRESS_KEY',how='left',validate='many_to_one')
rent_geo['BUILDING_AGE']=rent_geo['RENT_YEAR']-pd.to_numeric(rent_geo['YEAR_COMPLETED'],errors='coerce'); print('HDB match rate:',rent_geo['LATITUDE'].notna().mean())

exact duplicate rows: 1261
HDB match rate: 1.0


## Calculate nearest facilities with Haversine distance

Latitude/longitude are converted to radians and passed through the Haversine formula. Distances are multiplied by the Earth radius in metres. We calculate distances once per unique geocoded HDB block, retain the nearest facility name for diagnostics, and join the three distance features back to rentals.

In [19]:
def facilities(df):
    out=df.copy(); out['LATITUDE']=pd.to_numeric(out['LATITUDE'],errors='coerce'); out['LONGITUDE']=pd.to_numeric(out['LONGITUDE'],errors='coerce'); out=out.dropna(subset=['LATITUDE','LONGITUDE'])
    return out.loc[out['LATITUDE'].between(1.1,1.6)&out['LONGITUDE'].between(103.5,104.2)].reset_index(drop=True)
schools,malls,mrt=map(facilities,[schools_raw,malls_raw,mrt_raw])
def nearest(blocks,places,prefix):
    a=np.radians(blocks[['LATITUDE','LONGITUDE']].to_numpy(float)); b=np.radians(places[['LATITUDE','LONGITUDE']].to_numpy(float));
    dlat=b[:,0][None,:]-a[:,0][:,None]; dlon=b[:,1][None,:]-a[:,1][:,None]
    h=np.sin(dlat/2)**2+np.cos(a[:,0,None])*np.cos(b[:,0][None,:])*np.sin(dlon/2)**2
    distances=6371008.8*2*np.arcsin(np.sqrt(np.clip(h,0,1))); idx=distances.argmin(axis=1); rows=np.arange(len(blocks))
    return pd.DataFrame({f'NEAREST_{prefix}_NAME':places.iloc[idx]['NAME'].to_numpy(),f'NEAREST_{prefix}_M':distances[rows,idx]})
blocks=rent_geo[['ADDRESS_KEY','LATITUDE','LONGITUDE']].dropna().drop_duplicates('ADDRESS_KEY').reset_index(drop=True)
access=pd.concat([blocks[['ADDRESS_KEY']],nearest(blocks,schools,'SCHOOL'),nearest(blocks,malls,'MALL'),nearest(blocks,mrt,'MRT')],axis=1)
rent_geo=rent_geo.merge(access,on='ADDRESS_KEY',how='left',validate='many_to_one'); print(rent_geo[['NEAREST_SCHOOL_M','NEAREST_MALL_M','NEAREST_MRT_M']].describe())

       NEAREST_SCHOOL_M  NEAREST_MALL_M  NEAREST_MRT_M
count     149359.000000   149359.000000  149359.000000
mean         345.337459      810.376801     591.001254
std          211.725573      447.380848     326.894754
min           24.210155        0.025758      21.845373
25%          201.995769      462.501536     368.040163
50%          294.313634      732.251021     533.273210
75%          433.064544     1083.309823     756.583743
max         3309.220794     3202.447107    2571.686850


## Encode features

The model excludes identifiers, raw postal code, dates, target, and nearest-facility names. Distances, coordinates, floor values, building age, and match flags remain numeric and receive median imputation. Nominal fields such as town, flat type, subzone, planning area, and region are one-hot encoded; this avoids assigning a false ordering to categories. `handle_unknown='ignore'` makes the encoder safe for unseen test categories.

In [20]:
y=rent_geo.pop('MONTHLY_RENT').astype(float)
drop=['RENT_APPROVAL_DATE','ADDRESS_KEY','POSTAL_CODE','FEE','NEAREST_SCHOOL_NAME','NEAREST_MALL_NAME','NEAREST_MRT_NAME']
X=rent_geo.drop(columns=drop,errors='ignore').replace(r'^\s*$',np.nan,regex=True)
valid=y.notna(); X=X.loc[valid]; y=y.loc[valid]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
num=X_train.select_dtypes(include='number').columns.tolist(); cat=X_train.select_dtypes(exclude='number').columns.tolist()
preprocessor=ColumnTransformer([('numeric',SimpleImputer(strategy='median'),num),('categorical',OneHotEncoder(handle_unknown='ignore'),cat)],remainder='drop')
X_train_encoded=preprocessor.fit_transform(X_train); X_test_encoded=preprocessor.transform(X_test)
print('numeric columns:',num); print('categorical columns:',cat); print('encoded feature count:',X_train_encoded.shape[1])

numeric columns: ['FLOOR_AREA_SQM', 'LEASE_COMMENCE_DATE', 'RENT_YEAR', 'RENT_MONTH', 'LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'YEAR_COMPLETED', 'BUILDING_AGE', 'NEAREST_SCHOOL_M', 'NEAREST_MALL_M', 'NEAREST_MRT_M']
categorical columns: ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED', 'SUBZONE', 'PLANNING_AREA', 'REGION']
encoded feature count: 3548


## Train and evaluate XGBoost

This is a simple baseline using 300 trees, moderate depth, a small learning rate, and row/column subsampling. XGBoost can learn nonlinear effects and interactions between accessibility, floor, geography, and property features. MAE and RMSE are errors in rental dollars; higher R² is better.

## Explicit XGBoost training, testing, and evaluation

These cells separate model fitting, held-out prediction, and metric calculation.

In [ ]:
xgb_model_final = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror', eval_metric='rmse', random_state=42, n_jobs=2)
xgb_model_final.fit(X_train_encoded, y_train)
print('XGBoost training completed.')

In [ ]:
test_predictions = xgb_model_final.predict(X_test_encoded)
prediction_results = pd.DataFrame({'actual_rent': y_test.to_numpy(), 'predicted_rent': test_predictions})
prediction_results.head()

In [ ]:
evaluation_metrics = pd.DataFrame({'metric':['MAE','RMSE','R²'], 'value':[mean_absolute_error(y_test, test_predictions), np.sqrt(mean_squared_error(y_test, test_predictions)), r2_score(y_test, test_predictions)]})
evaluation_metrics